# 🔬 Chapter 15: Applications of Eigendecomposition and SVD (PCA)

---

## 15.1 Introduction: From Theory to Principal Components

Now that we possess the mathematical tools to decompose any matrix into its fundamental axes of variation (using Eigendecomposition for symmetric matrices and SVD for all matrices), we can tackle one of the biggest challenges in Data Science: the Curse of Dimensionality.

When datasets have hundreds or thousands of features, visualizing the data is impossible, and machine learning models often overfit. **Principal Component Analysis (PCA)** is a technique used to reduce the dimensionality of a dataset while retaining as much of the original variance (information) as possible.

PCA asks a simple geometric question: *"If I want to project this massive scatter plot of data onto a single line (or a 2D plane), which line captures the widest spread of the data?"*

In this chapter, we will prove that finding this line of maximum variance is exactly equivalent to finding the eigenvectors of the data's covariance matrix.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set plotting style
%matplotlib inline
np.random.seed(42)
print("Scientific libraries successfully imported for Chapter 15.")

## 15.2 PCA via Eigendecomposition (The Covariance Approach)

The traditional way to perform PCA relies heavily on Chapter 7 (The Covariance Matrix) and Chapter 13 (Eigendecomposition). The algorithm follows a strict recipe:

1. **Mean-Center the Data:** Subtract the mean of each column from the data. This shifts the entire scatter plot so its center of gravity rests perfectly on the origin `(0,0)`. 
2. **Compute the Covariance Matrix:** Multiply the mean-centered data matrix by its transpose: $\mathbf{C} = \frac{1}{n-1} \mathbf{X}^T \mathbf{X}$.
3. **Eigendecomposition:** Compute the eigenvalues and eigenvectors of the covariance matrix $\mathbf{C}$. 
4. **Sort:** Sort the eigenvectors in descending order based on their eigenvalues. The eigenvector with the largest eigenvalue is **Principal Component 1 (PC1)**, pointing in the direction of maximum variance.
5. **Project:** Multiply the original data by these eigenvectors to project the data into the new, lower-dimensional space.

In [ ]:
# 1. Create a synthetic 2D dataset with strong correlation (e.g., Height vs Weight)
n_samples = 200
x = np.random.randn(n_samples)
y = 2.5 * x + np.random.randn(n_samples) * 0.8
X = np.column_stack((x, y))

# 2. Mean-centering
X_centered = X - np.mean(X, axis=0)

# 3. Compute the Covariance Matrix
cov_matrix = (X_centered.T @ X_centered) / (n_samples - 1)

# 4. Perform Eigendecomposition on the Covariance Matrix
# Because covariance matrices are strictly symmetric, we use eigh()
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

# 5. Sort the eigen-pairs in descending order
sorted_idx = np.argsort(eigenvalues)[::-1]
eigenvalues_sorted = eigenvalues[sorted_idx]
eigenvectors_sorted = eigenvectors[:, sorted_idx]

print("Top Eigenvalue (Variance captured by PC1):", np.round(eigenvalues_sorted[0], 2))
print("Second Eigenvalue (Variance captured by PC2):", np.round(eigenvalues_sorted[1], 2))
print("\nPrincipal Component Vectors (Columns):\n", np.round(eigenvectors_sorted, 3))

## 15.3 PCA via Singular Value Decomposition (The Direct Approach)

While the covariance approach works, computing $\mathbf{X}^T \mathbf{X}$ on massive datasets can be numerically unstable and memory-intensive.

As we learned in Chapter 14, we can bypass the covariance matrix entirely by running **SVD directly on the mean-centered data matrix $\mathbf{X}$**.
$$ \mathbf{X} = \mathbf{U} \mathbf{\Sigma} \mathbf{V}^T $$

The rows of $\mathbf{V}^T$ (or the columns of $\mathbf{V}$) are exactly the Principal Components! We don't even need to sort them because SVD automatically sorts the singular values in descending order.

In [ ]:
# Perform SVD directly on the mean-centered data
U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)

# The Principal Components are the rows of Vt (or columns of V)
principal_components_svd = Vt.T

print("Principal Components via Covariance Eigendecomposition:\n", np.round(eigenvectors_sorted, 3))
print("\nPrincipal Components via Direct SVD:\n", np.round(principal_components_svd, 3))

# Note: Eigenvectors can have their signs flipped (multiplied by -1) and still be perfectly mathematically valid. 
# They represent the exact same line in space, just pointing in the opposite direction.

## 15.4 Visualizing the Principal Components

To truly understand PCA, we must visualize it. 
We will plot the mean-centered scatter data, and overlay the two Principal Component vectors as arrows originating from the center. 

You will immediately see that:
1. **PC1 (The largest vector)** points perfectly along the primary "corridor" of the data spread.
2. **PC2 (The smaller vector)** is perfectly orthogonal (90 degrees) to PC1, capturing the leftover "thickness" or noise in the data.

In [ ]:
plt.figure(figsize=(8, 8))

# Scatter the mean-centered data
plt.scatter(X_centered[:, 0], X_centered[:, 1], alpha=0.5, color='gray', label='Mean-Centered Data')

# Extract the vectors for plotting (scaled by their singular values for visual emphasis)
# S[0] and S[1] are proportional to the standard deviation along those axes
pc1 = principal_components_svd[:, 0] * (S[0] / np.sqrt(n_samples))
pc2 = principal_components_svd[:, 1] * (S[1] / np.sqrt(n_samples))

# Plot PC1
plt.quiver(0, 0, pc1[0], pc1[1], angles='xy', scale_units='xy', scale=1, 
           color='red', width=0.01, label='PC1 (Max Variance)')

# Plot PC2
plt.quiver(0, 0, pc2[0], pc2[1], angles='xy', scale_units='xy', scale=1, 
           color='blue', width=0.01, label='PC2 (Orthogonal Noise)')

plt.title('Principal Components Overlaid on Data Space')
plt.xlabel('Feature 1 (Centered)')
plt.ylabel('Feature 2 (Centered)')
plt.axhline(0, color='black', linestyle='--', alpha=0.3)
plt.axvline(0, color='black', linestyle='--', alpha=0.3)
plt.legend()
plt.gca().set_aspect('equal') # Forces the grid to be square so 90 degrees looks like 90 degrees
plt.grid(alpha=0.3)
plt.show()

## 15.5 Dimensionality Reduction (Projection)

Now for the final step. We have the new coordinate system (the Principal Components). To reduce the dimension of our data from 2D down to 1D, we simply take the dot product of our data matrix $\mathbf{X}$ and the first principal component vector $\mathbf{v}_1$.

$$ \mathbf{X}_{reduced} = \mathbf{X} \mathbf{v}_1 $$

This flattens the 2D scatter plot entirely onto the red arrow (PC1), saving 50% of the memory while keeping the vast majority of the statistical variance.

In [ ]:
# Select only the first Principal Component (Column 0)
pc1_vector = principal_components_svd[:, 0].reshape(-1, 1)

# Project the data onto PC1
X_projected = X_centered @ pc1_vector

print(f"Original Data Shape: {X_centered.shape} (2 Features)")
print(f"Projected Data Shape: {X_projected.shape} (1 Feature / 1 Principal Component)")

# Calculate the percentage of variance explained by PC1
variance_explained = (S[0]**2) / np.sum(S**2) * 100
print(f"\nBy keeping only 1 feature out of 2, we preserved {variance_explained:.2f}% of the total variance/information!")

## 15.6 Chapter Summary

In this application chapter, we demystified Principal Component Analysis (PCA):
- **The Goal:** To find a new, orthogonal coordinate system (a basis) where the first axis aligns with the direction of maximum variance in the data.
- **Method 1 (Covariance + Eigendecomposition):** We calculate $\mathbf{X}^T\mathbf{X}$ and find its eigenvectors. While mathematically correct, it can be computationally heavy and numerically unstable for large datasets.
- **Method 2 (SVD):** We perform SVD directly on the mean-centered data $\mathbf{X}$. The right singular vectors ($\mathbf{V}^T$) instantly give us our sorted principal components. This is the industry standard for algorithms under the hood of libraries like `scikit-learn`.
- **Projection:** By dot-multiplying the raw data by the top $k$ principal components, we squash the dataset into a lower dimension, filtering out noise and drastically improving computational speed for downstream machine learning tasks.